In [0]:
# Auto Loader bronze ingestion for the Customer 360 landing zone.
#
# Input layout (one CSV per batch; a date can carry multiple batches b0000, b0001, ...):
#   /Volumes/c360/landing/raw/<source>/<entity>/batch_date=YYYY-MM-DD/<entity>_bNNNN.csv
#
# Each business (source, entity) pair is ingested into c360.bronze_dev.<source>_<entity> and gets:
#   - _batch_data  (DATE, parsed from the batch_date=... folder name)
#   - _source_file (full path of the CSV the row came from)
#   - _loaded_at   (timestamp of the micro-batch that wrote the row)
#
# CSV columns are ingested as STRING (raw as received); only the added metadata
# columns (_batch_data, _source_file, _loaded_at) are typed.
#
# Each table is a Trigger.AvailableNow Auto Loader stream: it processes only files
# not yet seen (tracked by the checkpoint), so future batch dates and additional
# batch files land automatically on the next run of this notebook.

from pyspark.sql import functions as F

RAW_BASE = "/Volumes/c360/landing/raw"
CHECKPOINT_BASE = f"{RAW_BASE}/_checkpoints"
TARGET_SCHEMA = "c360.bronze_dev"

# (source dir, entity dir, bronze table name)
BRONZE_SOURCES = [
    ("billing", "customers", "billing_customers"),
    ("billing", "invoices", "billing_invoices"),
    ("crm", "contacts", "crm_contacts"),
    ("marketing", "leads", "marketing_leads"),
    ("marketing", "touches", "marketing_touches"),
    ("support", "requesters", "support_requesters"),
    ("support", "tickets", "support_tickets"),
]

# The answer key (source record -> true_customer_id) is ground truth for scoring
# matches, NOT modeling data. PROJECT_PLAN.md Phase 1: "keep the true_customer_id
# answer key out of bronze" — it is deliberately not a bronze table so the
# identity-resolution models cannot read true_customer_id. It ingests into its
# own schema, c360.evaluation_dev, with the same Auto Loader pattern and the
# SAME checkpoint as before the move (_checkpoints/answer_key), so files
# already seen are not re-read. Only the Phase-2 match_evaluation dbt model
# joins it (via source('evaluation', 'answer_key')).
ANSWER_KEY = ("_answer_key", "answer_key", "answer_key")  # -> c360.evaluation_dev.answer_key

def ingest_to_bronze(source_dir, entity_dir, table_name, target_schema=TARGET_SCHEMA):
    """Run a one-shot (availableNow) Auto Loader stream for one (source, entity) pair."""
    input_path = f"{RAW_BASE}/{source_dir}/{entity_dir}"
    checkpoint = f"{CHECKPOINT_BASE}/{table_name}"
    target = f"{target_schema}.{table_name}"

    def write_batch(df, batch_id):
        # txnAppId/txnVersion make reprocessed micro-batches idempotent on retries.
        (df.withColumn("_loaded_at", F.current_timestamp())
           .write.format("delta").mode("append")
           .option("txnAppId", f"c360_bronze_{table_name}")
           .option("txnVersion", batch_id)
           .saveAsTable(target))

    query = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", checkpoint)
        .option("cloudFiles.partitionColumns", "batch_date")
        .load(input_path)
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_batch_data", F.to_date("batch_date"))
        .drop("batch_date")
        .writeStream
        .foreachBatch(write_batch)
        .option("checkpointLocation", f"{checkpoint}/_stream")
        .trigger(availableNow=True)
        .queryName(f"bronze_{table_name}")
        .start()
    )
    query.awaitTermination()
    print(f"{target}: {spark.table(target).count()} rows")

for source_dir, entity_dir, table_name in BRONZE_SOURCES:
    ingest_to_bronze(source_dir, entity_dir, table_name)

# Answer key lands in the evaluation schema, never bronze (see note above).
ingest_to_bronze(*ANSWER_KEY, target_schema="c360.evaluation_dev")

In [0]:
def list_files(path, show_directories=False):
    """Recursively list everything under a Unity Catalog volume path.

    Args:
        path: Root path, e.g. "/Volumes/c360/landing/raw"
        show_directories: If True, include directory paths in the output;
                          if False (default), only files are returned.
    """
    results = []
    for entry in dbutils.fs.ls(path):
        if entry.isDir():
            if show_directories:
                results.append(entry.path)
            results.extend(list_files(entry.path, show_directories))
        else:
            results.append(entry.path)
    return results


files = list_files("/Volumes/c360/landing/raw", show_directories=False)
print(f"Found {len(files)} files")
print('\n'.join(files))

In [0]:
# Verify: per-table row counts, distinct batch dates, and audit columns.
# Covers the 7 bronze tables plus the answer key in c360.evaluation_dev.
from pyspark.sql import functions as F

all_tables = [(TARGET_SCHEMA, t) for _, _, t in BRONZE_SOURCES] + [("c360.evaluation_dev", "answer_key")]

summary = []
for schema, table_name in all_tables:
    r = (spark.table(f"{schema}.{table_name}")
         .agg(
             F.count("*").alias("rows"),
             F.countDistinct("_batch_data").alias("batch_dates"),
             F.min("_batch_data").alias("first_batch"),
             F.max("_batch_data").alias("latest_batch"),
             F.min("_loaded_at").alias("first_loaded_at"),
             F.max("_loaded_at").alias("latest_loaded_at"),
         )
         .collect()[0].asDict())
    summary.append((table_name, r["rows"], r["batch_dates"], r["first_batch"], r["latest_batch"], r["first_loaded_at"], r["latest_loaded_at"]))

display(spark.createDataFrame(
    summary,
    "table string, rows long, batch_dates long, first_batch date, latest_batch date, first_loaded_at timestamp, latest_loaded_at timestamp",
))

In [0]:
%sql
SELECT *
FROM c360.bronze_dev.billing_customers
LIMIT 10;